In [4]:
import ee
import geemap
import json
from IPython.display import Image, display

# Constants
SERVICE_ACCOUNT = '488436559965-compute@developer.gserviceaccount.com'
PROJECT_ID = 'avid-infinity-456706-d6'  # Your GCP project ID

# Authenticate and initialize Earth Engine with project ID
try:
    ee.Initialize(project=PROJECT_ID)
    print("✅ Earth Engine initialized successfully!")
except Exception as e:
    print(f"❌ Error initializing Earth Engine: {e}")
    print("Please run: earthengine authenticate")


✅ Earth Engine initialized successfully!


In [5]:
# Define POI polygon (your study area)
poi = ee.Geometry.Polygon([
    [77.77333199305133, 12.392392446684909],
    [77.77285377084087, 12.391034719901086],
    [77.77415744218291, 12.390603704636632],
    [77.77438732135664, 12.391302225016886],
    [77.77376792469431, 12.391501801924363],
    [77.77399141833513, 12.392187846379386],
    [77.77333199305133, 12.392392446684909]
])

# Date range
start_date = '2024-01-01'
end_date = '2024-12-31'

print(f"Study area defined")
print(f"Date range: {start_date} to {end_date}")




Study area defined
Date range: 2024-01-01 to 2024-12-31


In [17]:
# Load Sentinel-2 SR imagery with band selection to fix incompatible bands error
collection = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterBounds(poi) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .select(['B4', 'B8'])  # 🔧 FIX: Select only needed bands

# Check collection size
print(collection.first().getInfo())
collection_size = collection.size().getInfo()
print(f"📊 Found {collection_size} images in the collection")

if collection_size == 0:
    print("⚠️ No images found! Try expanding date range or increasing cloud percentage threshold.")
else:
    # Compute median image
    image = collection.median()
    print("✅ Median composite created successfully!")


{'type': 'Image', 'bands': [{'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32643', 'crs_transform': [10, 0, 699960, 0, -10, 1400040]}, {'id': 'B8', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32643', 'crs_transform': [10, 0, 699960, 0, -10, 1400040]}], 'version': 1740522676299983, 'id': 'COPERNICUS/S2_SR/20240118T051131_20240118T052357_T43PGP', 'properties': {'SPACECRAFT_NAME': 'Sentinel-2A', 'SATURATED_DEFECTIVE_PIXEL_PERCENTAGE': 0, 'BOA_ADD_OFFSET_B12': -1000, 'CLOUD_SHADOW_PERCENTAGE': 0.814947, 'system:footprint': {'type': 'LinearRing', 'coordinates': [[76.8410807013455, 12.658204357875261], [76.84107770957309, 12.658189624350761], [76.83759409359516, 12.162062647525616], [76.83426024068308, 11.665924516092655], [76.8342972878965, 11.665882801865887], [76.83432870322083, 11.665836917185592], [76.83434383889247, 11.6

In [19]:
# Extract generation time for all images in the collection
from datetime import datetime

print("📅 GENERATION TIMES FOR ALL IMAGES IN COLLECTION:")
print("=" * 60)

# Get collection size
collection_size = collection.size().getInfo()
print(f"�� Total images found: {collection_size}")

if collection_size > 0:
    # Get all images info
    collection_list = collection.toList(collection_size)
    
    # Extract generation time for each image
    for i in range(collection_size):
        image = collection_list.get(i)
        image_info = image.getInfo()
        
        # Extract image ID for reference
        image_id = image_info['id']
        
        # Extract generation time
        generation_time = image_info['properties']['GENERATION_TIME']
        generation_datetime = datetime.fromtimestamp(generation_time / 1000)
        
        # Extract acquisition date from image ID for comparison
        id_timestamp = image_id.split('/')[-1].split('_')[0]
        acquisition_datetime = datetime.strptime(id_timestamp, '%Y%m%dT%H%M%S')
        
        print(f"\n🖼️ Image {i+1}:")
        print(f"   📸 ID: {image_id.split('/')[-1]}")
        print(f"   📅 Acquisition: {acquisition_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   ⚙️ Generation: {generation_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   ⏱️ Processing Delay: {generation_datetime - acquisition_datetime}")

else:
    print("⚠️ No images found in the collection!")

print("\n" + "=" * 60)
print("�� SUMMARY:")
print(f"   • Date Range: {start_date} to {end_date}")
print(f"   • Images Found: {collection_size}")

📅 GENERATION TIMES FOR ALL IMAGES IN COLLECTION:
�� Total images found: 37

🖼️ Image 1:
   📸 ID: 20240118T051131_20240118T052357_T43PGP
   📅 Acquisition: 2024-01-18 05:11:31
   ⚙️ Generation: 2024-01-18 03:13:58
   ⏱️ Processing Delay: -1 day, 22:02:27

🖼️ Image 2:
   📸 ID: 20240118T051131_20240118T052357_T43PHP
   📅 Acquisition: 2024-01-18 05:11:31
   ⚙️ Generation: 2024-01-18 03:13:58
   ⏱️ Processing Delay: -1 day, 22:02:27

🖼️ Image 3:
   📸 ID: 20240123T051119_20240123T052221_T43PGP
   📅 Acquisition: 2024-01-23 05:11:19
   ⚙️ Generation: 2024-01-23 02:33:07
   ⏱️ Processing Delay: -1 day, 21:21:48

🖼️ Image 4:
   📸 ID: 20240123T051119_20240123T052221_T43PHP
   📅 Acquisition: 2024-01-23 05:11:19
   ⚙️ Generation: 2024-01-23 02:33:07
   ⏱️ Processing Delay: -1 day, 21:21:48

🖼️ Image 5:
   📸 ID: 20240207T051001_20240207T052026_T43PGP
   📅 Acquisition: 2024-02-07 05:10:01
   ⚙️ Generation: 2024-02-07 03:57:51
   ⏱️ Processing Delay: -1 day, 22:47:50

🖼️ Image 6:
   📸 ID: 20240207T0510

In [7]:
# MSAVI calculation
nir = image.select('B8')
red = image.select('B4')

# MSAVI formula: (2*NIR + 1 - sqrt((2*NIR + 1)^2 - 8*(NIR - Red))) / 2
msavi = nir.multiply(2).add(1) \
    .subtract((nir.multiply(2).add(1)).pow(2).subtract(nir.subtract(red).multiply(8)).sqrt()) \
    .divide(2).rename('MSAVI')

print("🌱 MSAVI calculated successfully!")


🌱 MSAVI calculated successfully!


In [8]:
# Reduce region to get min/max/mean statistics
stats = msavi.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=poi,
    scale=10,
    maxPixels=1e9
)

# Get statistics
stats_info = stats.getInfo()
print('📊 MSAVI Statistics:')
print(f"   Min: {stats_info.get('MSAVI_min', 'N/A'):.4f}")
print(f"   Max: {stats_info.get('MSAVI_max', 'N/A'):.4f}")
print(f"   Mean: {stats_info.get('MSAVI_mean', 'N/A'):.4f}")


📊 MSAVI Statistics:
   Min: 0.2471
   Max: 0.5187
   Mean: 0.4238


In [ ]:
# Create interactive map
Map = geemap.Map()

# Center map on your POI
Map.centerObject(poi, 16)

# Define visualization parameters for MSAVI
msavi_vis = {
    'min': 0.02,
    'max': 0.8,
    'palette': ['red', 'orange', 'yellow', 'lightgreen', 'green', 'darkgreen']
}

# Add layers to map
Map.addLayer(msavi.clip(poi), msavi_vis, 'MSAVI', True)
#Map.addLayer(poi, {'color': 'red', 'width': 2, 'fillColor': '00000000'}, 'Study Area')

# Add layer control and other tools
Map.add_layer_control()
Map.add_inspector()
Map.add_colorbar(msavi_vis)

print("🗺️ Interactive map created!")
Map


🗺️ Interactive map created!


Map(center=[12.391426405419157, 77.77359852088523], controls=(WidgetControl(options=['position', 'transparent_…